In [1]:
# For processing the timeseries
import pandas as pd, os, datetime
import numpy as np

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
sdate, edate = '2018-10-06','2019-04-26'

In [4]:
gen_details = pd.read_csv(f"{nmap_path}/nmap.csv")

In [17]:
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")
hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
hw_tseries = hw_tseries.set_index(['time']).sort_index()
hw_tseries = hw_tseries.loc[sdate:edate]
hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

In [6]:
def process_all(gen_details, gen_fpath, hw_tseries, start_date=sdate, end_date=edate):
    """
    Process all generator data in gen_details without grouping.
    
    Parameters:
        gen_details (pd.DataFrame): DataFrame containing at least 'duid' column.
        gen_fpath (str): Directory path containing CSV files named by DUID.
        hw_tseries (pd.DataFrame): Heatwave timeseries data with 'time' and 'DUID'.
        start_date (str): Start date for filtering time series.
        end_date (str): End date for filtering time series.
    
    Returns:
        pd.DataFrame: Merged dataframe with summed TOTALMWh per DUID per hour and heatwave info.
    """
    # Build file paths for all DUIDs
    gen_locs = gen_fpath + '/' + gen_details['duid'] + ".csv"

    # Read all existing CSVs
    dfs = [pd.read_csv(fp) for fp in gen_locs if os.path.exists(fp)]
    
    if not dfs:
        print("[WARN] No data files found for any DUID.")
        return pd.DataFrame()  # Return empty if no files
    
    # Concatenate all data
    dfs = pd.concat(dfs, ignore_index=True)

    # Convert time to datetime and filter
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    # Group by DUID and hourly time, sum TOTALMWh
    grouped = dfs.groupby(['DUID', pd.Grouper(freq='1h')])['TOTALMWh'].sum().reset_index()

    # Sort hw_tseries once
    hw_sorted = hw_tseries.sort_values(by=['time', 'DUID'])

    # Merge with heatwave timeseries
    merged = pd.merge_asof(
        grouped.sort_values(by=['time', 'DUID']),
        hw_sorted,
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("1d"),
        direction='nearest'
    )
    
    return merged

In [7]:
df = process_all(gen_details, gen_fpath, hw_tseries)
df = df.merge(gen_details[['duid', 'region','fuel_source_primary']], left_on='DUID', right_on='duid', how='left').drop(columns='duid')

# del hw_tseries

In [8]:
def agg_group(df,grouping):
    agg_func = {'TOTALMWh':'sum','EHF_flag':'max', 'EHF_val':'max','HW_event_day':'first'}
    grp = df.groupby([grouping, 'time']).aggregate(agg_func).reset_index()
    
    grp['normalised'] = grp.groupby(grouping)['TOTALMWh'].transform(
                            lambda x: (x - x.min()) / (x.max() - x.min()))
    return grp

In [9]:
def highLights(df, fig, variable, level, mode, fillcolor, layer):
    """
    Set a specified color as background for given
    levels of a specified variable using a shape.
    
    Keyword arguments:
    ==================
    fig -- plotly figure
    variable -- column name in a pandas dataframe
    level -- int or float
    mode -- set threshold above or below
    fillcolor -- any color type that plotly can handle
    layer -- position of shape in plotly fiugre, like "below"
    
    """
    
    if mode == 'above':
        m = df[variable].gt(level)
    
    if mode == 'below':
        m = df[variable].lt(level)
        
    df1 = df[m].groupby((~m).cumsum())['time'].agg(['first','last'])

    for index, row in df1.iterrows():
        #print(row['first'], row['last'])
        fig.add_shape(
            type="rect",
            xref="x",
            yref="paper",
            x0=row['first'],
            y0=0,
            x1=row['last'],
            y1=1,
            line=dict(color="rgba(0,0,0,0)", width=3),
            fillcolor="rgba(100,100,100,0.2)",
            layer=layer
        )
    return(fig)


In [10]:
def plot_agg_group(
    grouped_df, group_col, title='Time Series Plot', y='TOTALMWh',
    highlight=False, highlight_mode='union'
):
    fig = go.Figure()

    for group_name, group in grouped_df.groupby(group_col):
        fig.add_trace(go.Scatter(
            x=group['time'],
            y=group[y],
            mode='lines',
            name=str(group_name)
        ))

    if highlight:
        if highlight_mode == 'union':
            # Highlight where any group has EHF_flag==1 (union)
            highlight_times = (
                grouped_df.groupby('time')['EHF_flag']
                .max()
                .reset_index()
            )
            fig = highLights(
                df=highlight_times,
                fig=fig,
                variable='EHF_flag',
                level=0,
                mode='above',
                fillcolor='rgba(255,0,0,0.1)',
                layer='below'
            )
        elif highlight_mode == 'per_group':
            # Highlight per group
            for group_name, group in grouped_df.groupby(group_col):
                fig = highLights(
                    df=group,
                    fig=fig,
                    variable='EHF_flag',
                    level=0,
                    mode='above',
                    fillcolor='rgba(255,0,0,0.1)',
                    layer='below'
                )
        else:
            raise ValueError("highlight_mode must be 'union' or 'per_group'")

    fig.update_layout(
        title=title,
        xaxis_title='Time',
        yaxis_title='Total MWh',
        template='plotly_white',
        legend_title=group_col
    )

    # Add range slider
    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=7,
                         label="1w",
                         step="day",
                         stepmode="backward"),
                    dict(count=1,
                         label="1m",
                         step="month",
                         stepmode="backward"),
                    dict(step="all")
                ])
            ),
            rangeslider=dict(
                visible=True
            ),
            type="date"
        )
    )

    return fig

In [11]:
state_df = df.groupby('region')    
states = {name: group for name, group in state_df}
qld_df,nsw_df,vic_df,sa_df,tas_df = states['QLD1'],states['NSW1'],states['VIC1'],states['SA1'],states['TAS1']
del state_df,states

In [12]:
def split_state_level(df,region,by_fuel=False):
    state_df = df.groupby('region')    
    states = {name: group for name, group in state_df}
    state = states[region]
    if by_fuel == True:
        agg_func = {'TOTALMWh':'sum','EHF_flag':'max', 'EHF_val':'max','HW_event_day':'first'}
        state = state.groupby(['fuel_source_primary', 'time']).aggregate(agg_func).reset_index()
        state['normalised'] = state.groupby('fuel_source_primary')['TOTALMWh'].transform(
                        lambda x: (x - x.min()) / (x.max() - x.min()))
    return state      


In [13]:
demand = pd.read_csv('/scratch/ng72/ms5578/time_series/state_demand.csv')
demand['time'] = pd.to_datetime(demand['time'])
demand = demand.set_index('time').sort_index()
demand = demand.loc[sdate:edate]
demand = demand.reset_index()

agg_df = demand.groupby('time', as_index=False).agg({'TOTALDEMAND': 'sum'})
total_row = pd.DataFrame({
    'REGIONID': ['ALL'],
    'TOTALDEMAND': [agg_df['TOTALDEMAND'].sum()]
})
demand = pd.concat([demand, total_row]).set_index('time')

demand = demand.groupby([pd.Grouper(freq='1h'), 'REGIONID']).sum().reset_index()#.set_index('time')
del agg_df, total_row

In [14]:
qld_df = split_state_level(df,'QLD1',by_fuel=False)
qld_df = agg_group(qld_df,'region')

In [15]:
qld_demand = demand[demand["REGIONID"] == 'QLD1'][['time', 'TOTALDEMAND']]
qld_demand['dem_normalised'] = qld_demand['TOTALDEMAND'].transform(
                        lambda x: (x - x.min()) / (x.max() - x.min()))
qld_df = qld_df.merge(qld_demand, on='time', how='left')
qld_df['sub_dem'] = qld_df['normalised'] - qld_df['dem_normalised']

In [21]:
new_df = hw_tseries.groupby('DUID')['EHF_flag'].apply(lambda x: (x == 1).sum()).reset_index(name='days_in_HW')
new_df

,DUID,days_in_HW
0,ADPBA1G,18
1,ADPBA1L,18
2,ADPMH1,18
3,ADPPV1,18
4,ADPPV2,18
...,...,...
482,YWNGAHYD,41
483,YWPS1,28
484,YWPS2,28
485,YWPS3,28
